# DuckDB SQL

In [2]:
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
minio_user = os.getenv("MINIO_ROOT_USER", "minioadmin")
minio_password = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE SECRET IF NOT EXISTS (
        TYPE S3,
        KEY_ID '{minio_user}',
        SECRET '{minio_password}',
        ENDPOINT 'localhost:9000',
        URL_STYLE 'path',
        USE_SSL false
    );
""")
print("Connected to DuckDB and MinIO successfully")

Connected to DuckDB and MinIO successfully


# Basic sqls

In [3]:
# query ot count total no of rows in whole file
q1 = """
SELECT COUNT(*) as total_rows from read_parquet('s3://analytics-data/ml_features.parquet')"""
res1 = con.execute(q1).df()
display(res1)

,total_rows
0,515952


In [4]:
# query to describe the cols in the file 
q2 = """ DESCRIBE SELECT * FROM read_parquet('s3://analytics-data/ml_features.parquet')  """
res2 = con.execute(q2).df()
display(res2)

,column_name,column_type,null,key,default,extra
0,asset_symbol,VARCHAR,YES,None,None,None
1,asset_class,VARCHAR,YES,None,None,None
2,exchange,VARCHAR,YES,None,None,None
3,interval,VARCHAR,YES,None,None,None
4,date,TIMESTAMP,YES,None,None,None
5,open,DOUBLE,YES,None,None,None
6,high,DOUBLE,YES,None,None,None
7,low,DOUBLE,YES,None,None,None
8,close,DOUBLE,YES,None,None,None
9,volume,DOUBLE,YES,None,None,None


In [5]:
# query to read parquet from ml_features where to read specific asset on specific interval 
q3 = """
SELECT date, asset_symbol, open, high, low, close, volume
FROM read_parquet('s3://analytics-data/ml_features.parquet')
WHERE asset_symbol = 'AAPL' AND interval = '1h'
ORDER BY date ASC
LIMIT 5;
"""
res3 = con.execute(q3).df()
display(res3)

,date,asset_symbol,open,high,low,close,volume
0,2024-05-13 16:30:00,AAPL,186.289993,186.929993,186.250000,186.658600,8098301.0
1,2024-05-13 17:30:00,AAPL,186.660004,186.949997,186.369995,186.650101,5453676.0
2,2024-05-13 18:30:00,AAPL,186.659897,187.100006,186.559998,186.989899,6516486.0
3,2024-05-13 19:30:00,AAPL,186.985001,187.029999,186.070007,186.285004,9250536.0
4,2024-05-14 13:30:00,AAPL,187.649994,188.300003,186.550003,187.009995,14754549.0


In [6]:
#query to get total no of rows in asset aapl for specific interval 
q4 = """
SELECT count(*) as total_rows 
FROM read_parquet('s3://analytics-data/ml_features.parquet') 
WHERE asset_symbol = 'AAPL'AND interval = '1h';
"""
res4 = con.execute(q4).df()
display(res4)

,total_rows
0,3207


# Advanced Sqls

# Moving AVG using window functions

In [7]:
# query for 7 day moving avg 

q5 = """ 
SELECT date,asset_symbol,close,
AVG(close) OVER (
PARTITION BY asset_symbol 
ORDER BY date asc
ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
) AS mv_avg_7_days
FROM read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'AAPL'
and interval = '1d'
order by date desc
limit 10
"""

res5 = con.execute(q5).df()
display(res5)


,date,asset_symbol,close,mv_avg_7_days
0,2026-03-17,AAPL,254.229996,256.471427
1,2026-03-16,AAPL,252.820007,256.852855
2,2026-03-13,AAPL,250.690002,257.919998
3,2026-03-12,AAPL,255.759995,259.609996
4,2026-03-11,AAPL,260.809998,260.751426
5,2026-03-10,AAPL,261.109985,261.309998
6,2026-03-09,AAPL,259.880005,261.748570
7,2026-03-06,AAPL,256.899994,263.615714
8,2026-03-05,AAPL,260.290009,266.091431
9,2026-03-04,AAPL,262.519989,267.784289


In [8]:
# today close and yesterday close  

q6 = """
select date,asset_symbol, close as todays_close,
lag(close,1) over (partition by asset_symbol order by date asc) as yesterday_close
FROM read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'AAPL'
and interval = '1d'
order by date desc
limit 10
"""
res6 = con.execute(q6).df()
display(res6)

,date,asset_symbol,todays_close,yesterday_close
0,2026-03-17,AAPL,254.229996,252.820007
1,2026-03-16,AAPL,252.820007,250.690002
2,2026-03-13,AAPL,250.690002,255.759995
3,2026-03-12,AAPL,255.759995,260.809998
4,2026-03-11,AAPL,260.809998,261.109985
5,2026-03-10,AAPL,261.109985,259.880005
6,2026-03-09,AAPL,259.880005,256.899994
7,2026-03-06,AAPL,256.899994,260.290009
8,2026-03-05,AAPL,260.290009,262.519989
9,2026-03-04,AAPL,262.519989,263.750000


In [9]:
# query for day over day percentage change 
q7 = """
select date,asset_symbol, close as todays_close,
lag(close,1) over (partition by asset_symbol order by date asc) as yesterday_close,
((close - LAG(close, 1) OVER (PARTITION BY asset_symbol ORDER BY date ASC)) / LAG(close, 1) OVER (PARTITION BY asset_symbol ORDER BY date ASC)) * 100 AS daily_percent_change
FROM read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'AAPL'
and interval = '1d'
order by date desc
limit 10
"""

res7 = con.execute(q7).df()
display(res7)



,date,asset_symbol,todays_close,yesterday_close,daily_percent_change
0,2026-03-17,AAPL,254.229996,252.820007,0.557704
1,2026-03-16,AAPL,252.820007,250.690002,0.849657
2,2026-03-13,AAPL,250.690002,255.759995,-1.982324
3,2026-03-12,AAPL,255.759995,260.809998,-1.936277
4,2026-03-11,AAPL,260.809998,261.109985,-0.114889
5,2026-03-10,AAPL,261.109985,259.880005,0.473288
6,2026-03-09,AAPL,259.880005,256.899994,1.159989
7,2026-03-06,AAPL,256.899994,260.290009,-1.302399
8,2026-03-05,AAPL,260.290009,262.519989,-0.849452
9,2026-03-04,AAPL,262.519989,263.750000,-0.466355


# Group BY ROLLUP, GROUPING SETS, CUBE

In [10]:
# rollup 
q8 = """
select asset_symbol,interval,sum(volume) as total_volume_traded
FROM read_parquet('s3://analytics-data/ml_features.parquet')
group by rollup (asset_symbol,interval)
order by asset_symbol, interval
"""

res8 = con.execute(q8).df()
display(res8)

,asset_symbol,interval,total_volume_traded
0,1000PEPE,1d,4.258446e+13
1,1000PEPE,1h,7.852455e+13
2,1000PEPE,None,1.211090e+14
3,AAPL,1d,5.078414e+11
4,AAPL,1h,1.798253e+10
...,...,...,...
63,XRP,1d,7.199967e+11
64,XRP,1h,7.660851e+11
65,XRP,W,1.534002e+11
66,XRP,None,1.639482e+12


In [11]:
# grouing sets 
q9 = """
select asset_symbol,interval,sum(volume) as total_volume_traded
FROM read_parquet('s3://analytics-data/ml_features.parquet')
group by grouping sets (asset_symbol,interval)
order by asset_symbol, interval
"""

res9 = con.execute(q9).df()
display(res9)

,asset_symbol,interval,total_volume_traded
0,1000PEPE,None,1.211090e+14
1,AAPL,None,7.921458e+11
2,ADA,None,8.668396e+11
3,AMZN,None,4.499194e+11
4,AVAX,None,1.323265e+10
5,BTC,None,4.900716e+08
6,DOGE,None,6.819111e+12
7,DOT,None,3.259537e+10
8,DYDX,None,6.381132e+10
9,ETH,None,4.303237e+09


In [12]:
# cube 
q10 = """
select asset_symbol,interval,sum(volume) as total_volume_traded
FROM read_parquet('s3://analytics-data/ml_features.parquet')
group by cube (asset_symbol,interval)
order by asset_symbol, interval
"""

res10 = con.execute(q10).df()
display(res10)

,asset_symbol,interval,total_volume_traded
0,1000PEPE,1d,4.258446e+13
1,1000PEPE,1h,7.852455e+13
2,1000PEPE,None,1.211090e+14
3,AAPL,1d,5.078414e+11
4,AAPL,1h,1.798253e+10
...,...,...,...
67,None,1d,4.809255e+13
68,None,1h,8.280080e+13
69,None,1wk,9.691539e+11
70,None,W,1.240188e+12


# Ranking 

In [13]:
# row number , rank and dense rank 
q11 = """
select date,asset_symbol,round(close, 0) as rounded_close,
row_number() over (
partition by asset_symbol
order by round(close, 0) desc
)as row_num,

rank() over (
partition by asset_symbol
order by round(close, 0) desc
)as normal_rank,

dense_rank() over (
partition by asset_symbol
order by round(close, 0) desc
)as dense_rank

FROM read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'AAPL'
and interval = '1d'
order by rounded_close desc
limit 15
"""

res11 = con.execute(q11).df()
display(res11)

,date,asset_symbol,rounded_close,row_num,normal_rank,dense_rank
0,2025-12-02,AAPL,286.0,1,1,1
1,2025-12-03,AAPL,284.0,2,2,2
2,2025-12-01,AAPL,283.0,3,3,3
3,2025-12-04,AAPL,280.0,4,4,4
4,2025-11-28,AAPL,279.0,7,5,5
5,2025-12-05,AAPL,279.0,6,5,5
6,2025-12-10,AAPL,279.0,5,5,5
7,2026-02-06,AAPL,278.0,8,8,6
8,2025-12-11,AAPL,278.0,10,8,6
9,2025-12-12,AAPL,278.0,11,8,6


# ROW_NUMBER()

In [14]:
# row_number()
q12 = """
with rankedvolume as (
select date, asset_symbol,volume,
row_number() over (
partition by asset_symbol
order by volume desc
)as volume_rank
FROM read_parquet('s3://analytics-data/ml_features.parquet')
where interval = '1d'
)

select date,asset_symbol,volume,volume_rank
from rankedvolume
where volume_rank <= 2
order by asset_symbol,volume_rank;
"""

res12 = con.execute(q12).df()
display(res12)

,date,asset_symbol,volume,volume_rank
0,2024-02-27,1000PEPE,5.340902e+11,1
1,2024-02-28,1000PEPE,5.258611e+11,2
2,2013-01-24,AAPL,1.460852e+09,1
3,2012-11-16,AAPL,1.266894e+09,2
4,2025-03-03,ADA,2.732464e+09,1
5,2025-03-02,ADA,2.385510e+09,2
6,2015-01-30,AMZN,4.771220e+08,1
7,2015-07-24,AMZN,4.381880e+08,2
8,2023-11-16,AVAX,2.795229e+07,1
9,2023-11-17,AVAX,2.716913e+07,2


# sum() 

In [15]:
# total cumulative volume using SUM()
q13 = """
select date,asset_symbol,volume as daily_volume,
SUM(volume) OVER (
PARTITION BY asset_symbol 
ORDER BY date ASC
)as cumulative_volume
FROM read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'AAPL' 
and interval = '1d'
order by date asc
limit 15;
"""

res13 = con.execute(q13).df()
display(res13)


,date,asset_symbol,daily_volume,cumulative_volume
0,2012-10-16,AAPL,5.497716e+08,5.497716e+08
1,2012-10-17,AAPL,3.890376e+08,9.388092e+08
2,2012-10-18,AAPL,4.766244e+08,1.415434e+09
3,2012-10-19,AAPL,7.440860e+08,2.159520e+09
4,2012-10-22,AAPL,5.467308e+08,2.706250e+09
5,2012-10-23,AAPL,7.071456e+08,3.413396e+09
6,2012-10-24,AAPL,5.585272e+08,3.971923e+09
7,2012-10-25,AAPL,6.563256e+08,4.628249e+09
8,2012-10-26,AAPL,1.018433e+09,5.646682e+09
9,2012-10-31,AAPL,5.100032e+08,6.156685e+09


In [16]:
# using date_trunc() , group by date and extraact()
q14 = """
select date_trunc('month', date) as trading_month,asset_symbol,sum(volume) as total_monthly_volume
from read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'AAPL' 
and interval = '1d'
and extract(year from date) = 2020 
group by trading_month, asset_symbol
order by trading_month asc;
"""
res14 = con.execute(q14).df()
display(res14)

,trading_month,asset_symbol,total_monthly_volume
0,2020-01-01,AAPL,2.934370e+09
1,2020-02-01,AAPL,3.019279e+09
2,2020-03-01,AAPL,6.280072e+09
3,2020-04-01,AAPL,3.265299e+09
4,2020-05-01,AAPL,2.805936e+09
5,2020-06-01,AAPL,3.243376e+09
6,2020-07-01,AAPL,3.020283e+09
7,2020-08-01,AAPL,4.070061e+09
8,2020-09-01,AAPL,3.885245e+09
9,2020-10-01,AAPL,2.894666e+09


# AVG() and HAVING

In [17]:
# avg and volume gettign avg daily colume but only greater tahn 1000000
q15 = """
select asset_symbol, avg(volume) as avg_daily_volume
from read_parquet('s3://analytics-data/ml_features.parquet')
where interval = '1d'
group by asset_symbol
having avg(volume) > 1000000
order by avg_daily_volume desc;
"""
res15 = con.execute(q15).df()
display(res15)

,asset_symbol,avg_daily_volume
0,1000PEPE,5.004050e+10
1,DOGE,1.884353e+09
2,XRP,4.583047e+08
3,ADA,2.150543e+08
4,AAPL,1.506054e+08
5,TSLA,1.093589e+08
6,AMZN,7.247627e+07
7,GOOGL,4.123097e+07
8,MSFT,3.070967e+07
9,META,2.640363e+07


# QUALIFY()

In [18]:
# new way instead of using having the qualify keyword for modern data engineering databases
q16 = """
select date, asset_symbol,volume, row_number() over (
partition by asset_symbol
order by volume desc
)as volume_rank
from read_parquet('s3://analytics-data/ml_features.parquet')
where interval = '1d'
qualify volume_rank <= 2
order by asset_symbol,volume_rank;
"""
res16 = con.execute(q16).df()
display(res16)

,date,asset_symbol,volume,volume_rank
0,2024-02-27,1000PEPE,5.340902e+11,1
1,2024-02-28,1000PEPE,5.258611e+11,2
2,2013-01-24,AAPL,1.460852e+09,1
3,2012-11-16,AAPL,1.266894e+09,2
4,2025-03-03,ADA,2.732464e+09,1
5,2025-03-02,ADA,2.385510e+09,2
6,2015-01-30,AMZN,4.771220e+08,1
7,2015-07-24,AMZN,4.381880e+08,2
8,2023-11-16,AVAX,2.795229e+07,1
9,2023-11-17,AVAX,2.716913e+07,2


# Views and Joins 

In [19]:
con.execute("""
create or replace view daily_avg_volume_view as 
select 
asset_symbol,avg(volume) as avg_daily_volume
from read_parquet('s3://analytics-data/ml_features.parquet')
where interval = '1d'
group by asset_symbol;
""")
q17 = """
select * from daily_avg_volume_view
where avg_daily_volume > 10000000 
order by avg_daily_volume desc;
"""
res17 = con.execute(q17).df()
display(res17)

,asset_symbol,avg_daily_volume
0,1000PEPE,5.004050e+10
1,DOGE,1.884353e+09
2,XRP,4.583047e+08
3,ADA,2.150543e+08
4,AAPL,1.506054e+08
5,TSLA,1.093589e+08
6,AMZN,7.247627e+07
7,GOOGL,4.123097e+07
8,MSFT,3.070967e+07
9,META,2.640363e+07


# UNION ALL 

In [20]:
q18 = """
(select date, asset_symbol, volume
from read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'AAPL' and interval = '1d'
order by volume desc
limit 5)
union all
(select date, asset_symbol, volume
from read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'BTC' and interval = '1d'
order by volume desc
limit 5);
"""
res18 = con.execute(q18).df()
display(res18)

,date,asset_symbol,volume
0,2013-01-24,AAPL,1.460852e+09
1,2012-11-16,AAPL,1.266894e+09
2,2013-01-25,AAPL,1.208026e+09
3,2012-12-06,AAPL,1.177212e+09
4,2014-01-28,AAPL,1.065523e+09
5,2022-11-08,BTC,6.995124e+05
6,2022-06-13,BTC,5.645824e+05
7,2024-08-05,BTC,5.630877e+05
8,2022-11-09,BTC,5.587211e+05
9,2022-06-15,BTC,5.296155e+05


# CORR correlation 

In [21]:
q20 = """
with apple_data as (
select date_trunc('day', date) as daily, log_returns as aapl_return
from read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'AAPL' and interval = '1d'
),
bitcoin_data as (
select date_trunc('day', date) as daily, log_returns as btc_return
from read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol = 'BTC' and interval = '1d'  -- Using BTC!
)
select round(corr(apple_data.aapl_return, bitcoin_data.btc_return), 4) as aapl_btc_correlation
from apple_data inner join bitcoin_data on apple_data.daily = bitcoin_data.daily;
"""
res20 = con.execute(q20).df()
display(res20)


,aapl_btc_correlation
0,0.2431


# Distinct

In [24]:
q21 = """
select distinct asset_symbol
from read_parquet('s3://analytics-data/ml_features.parquet')
where asset_symbol like 'A%';
"""
res21 = con.execute(q21).df()
display(res21)


,asset_symbol
0,AMZN
1,ADA
2,AAPL
3,AVAX
